In [2]:
# ========================================================
# Project: Housing Price Prediction Benchmarking (AutoML)
# Author: BOUZID Mohamed El Khallil
# Department: Organic Process Engineering
# ========================================================

# [1] Environment Setup
# Installing necessary AutoML frameworks
!pip install h2o auto-sklearn tpot mljar-supervised scikit-learn pandas numpy -q

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# [2] Data Preparation
print("Loading California Housing Dataset...")
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

# Split: 80% Training, 20% Testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

results_summary = {}

# [3] Framework 1: H2O AutoML
import h2o
from h2o.automl import H2OAutoML

print("\n--- Running H2O AutoML ---")
h2o.init(verbose=False)

hf_train = h2o.H2Frame(pd.concat([X_train, y_train], axis=1))
hf_test = h2o.H2Frame(pd.concat([X_test, y_test], axis=1))

# Limited to 5 models for demonstration speed
aml = H2OAutoML(max_models=5, seed=42, max_runtime_secs=180)
aml.train(x=list(X.columns), y='target', training_frame=hf_train)

lb = aml.leaderboard
print(lb.head(rows=3))

preds = aml.predict(hf_test).as_data_frame()
mse_h2o = mean_squared_error(y_test, preds)
results_summary['H2O'] = mse_h2o

# [4] Framework 2: Auto-sklearn
import autosklearn.regression

print("\n--- Running Auto-sklearn ---")
automl_sklearn = autosklearn.regression.AutoSklearnRegressor(
    time_left_for_this_task=180,
    per_run_time_limit=30,
    memory_limit=None
)
automl_sklearn.fit(X_train, y_train)
y_pred_sklearn = automl_sklearn.predict(X_test)
mse_sklearn = mean_squared_error(y_test, y_pred_sklearn)
results_summary['Auto-sklearn'] = mse_sklearn

# [5] Framework 3: TPOT
from tpot import TPOTRegressor

print("\n--- Running TPOT ---")
# Using low generation count for quick experimentation
tpot = TPOTRegressor(generations=2, population_size=10, verbosity=2, random_state=42)
tpot.fit(X_train, y_train)
y_pred_tpot = tpot.predict(X_test)
mse_tpot = mean_squared_error(y_test, y_pred_tpot)
results_summary['TPOT'] = mse_tpot

# [6] Framework 4: MLJAR
from supervised.automl import AutoML

print("\n--- Running MLJAR ---")
automl_mljar = AutoML(mode="Explain")
automl_mljar.fit(X_train, y_train)
y_pred_mljar = automl_mljar.predict(X_test)
mse_mljar = mean_squared_error(y_test, y_pred_mljar)
results_summary['MLJAR'] = mse_mljar

# [7] Final Benchmarking
print("\n========================================")
print("FINAL MSE RESULTS (Lower is better)")
print("========================================")
df_res = pd.DataFrame(list(results_summary.items()), columns=['Framework', 'MSE'])
df_res = df_res.sort_values(by='MSE').reset_index(drop=True)
print(df_res)
print("========================================")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (pyproject.toml) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
Loading California Housing Dataset...


ModuleNotFoundError: No module named 'h2o'